# 02 - Baseline Training

**Purpose.** Establish a small, fast baseline before spending hours on ImageNet backbones.

**Research integrity rule.** A baseline is not a side quest; it is the first defense against inflated ensemble claims.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT = Path('..').resolve()
DATA_DIR = PROJECT / 'data' / 'idc'
OUTPUT_DIR = PROJECT / 'artifacts' / 'notebook_simple_cnn'

print('project:', PROJECT)
print('data exists:', DATA_DIR.exists())
print('output:', OUTPUT_DIR)

## Run the Baseline

The cell below is guarded by `RUN_TRAINING = False` so opening the notebook does not unexpectedly start training. Set it to `True` when the dataset is ready.

In [ ]:
RUN_TRAINING = False

cmd = [
    'uv', 'run', 'dcpgann-train', str(DATA_DIR),
    '--config', str(PROJECT / 'configs' / 'paper_2022_idc.json'),
    '--epochs', '1',
    '--backbones', 'simple_cnn',
    '--output', str(OUTPUT_DIR),
]

print(' '.join(cmd))
if RUN_TRAINING:
    if not DATA_DIR.exists():
        raise FileNotFoundError('Prepare data/idc first; see README.md.')
    subprocess.run(cmd, cwd=PROJECT, check=True)
else:
    print('Set RUN_TRAINING = True to execute.')

## Inspect the Baseline Result

After running, the report should contain split sizes, class balance, individual model metrics, equal-weight ensemble metrics, and optimized ensemble metrics.

In [ ]:
experiment_path = OUTPUT_DIR / 'experiment.json'
if not experiment_path.exists():
    print('No baseline artifact yet:', experiment_path)
else:
    report = json.loads(experiment_path.read_text())
    print('split sizes:', report['split_sizes'])
    print('test metrics:')
    for key, value in report['optimized_ensemble_test_metrics'].items():
        print(f'  {key}: {value:.4f}' if isinstance(value, float) else f'  {key}: {value}')

## What to Look For

- The code should run end-to-end and produce artifacts.
- Metrics should be plausible, not necessarily high after one epoch.
- If the baseline is broken, do not start a full four-backbone run yet.